# 0. Carga de librerías

In [3]:
import os
import random
import itertools
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import kagglehub
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from PIL import Image, UnidentifiedImageError

# Reproducibilidad
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
keras.utils.set_random_seed(SEED)

# Parámetros globales
IMG_SIZE = 299
BATCH_SIZE = 32
NUM_CLASSES = 9

# 1. Carga de datos

In [ ]:
import pathlib

# Buscar la raíz del proyecto (contiene data.dvc) desde el directorio actual
_cwd = pathlib.Path().resolve()
_root = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / "realwaste_mod.dvc").exists()),
    _cwd
)
path = str(_root / "realwaste_mod")

categories = sorted(os.listdir(path))
rows = []
for category in categories:
    category_dir = os.path.join(path, category)
    for img_name in os.listdir(category_dir):
        rows.append({'image_path': os.path.join(category_dir, img_name), 'label': category})

df = pd.DataFrame(rows)
print(f"Total de imágenes: {len(df)}")
print(f"Clases ({df['label'].nunique()}): {categories}")

Total de imágenes: 4751
Clases (9): ['1-Cardboard', '2-Food Organics', '3-Glass', '4-Metal', '5-Miscellaneous Trash', '6-Paper', '7-Plastic', '8-Textile Trash', '9-Vegetation']


In [6]:
# Formato de las imágenes
def get_file_formats(folder_path):
    file_formats = set()

    for root, _, files in os.walk(folder_path):
        for filename in files:
            file_extension = os.path.splitext(filename)[1].lower()
            file_formats.add(file_extension)

    return file_formats

def get_image_size(folder_path):
    sizes = set()
    for root, _, files in os.walk(folder_path):
        for filename in files:
            file_path = os.path.join(root, filename)
            try:
                with Image.open(file_path) as img:
                  img_array = np.array(img)
                  sizes.add(img_array.shape)
            except UnidentifiedImageError:
                print(f"Archivo no reconocido como imagen: {file_path}")
    return sizes

file_formats = get_file_formats(path)
print(f"Formatos de archivo en la carpeta de imágenes: {', '.join(file_formats)}")
image_sizes = get_image_size(path)
print(f"Tamaños de imágenes: {', '.join([str(size) for size in image_sizes])}")

Formatos de archivo en la carpeta de imágenes: .jpg
Tamaños de imágenes: (299, 299, 3)


## 1.1 División en entrenamiento-validación-prueba

In [7]:
# Mapeo de etiquetas a índices numéricos
label_to_index = {label: i for i, label in enumerate(categories)}
print("Mapeo de etiquetas:", label_to_index)

# Partición estratificada 70 / 15 / 15
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df['label'], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED
)

print(f"\nEntrenamiento : {len(train_df)} imágenes")
print(f"Validación    : {len(val_df)} imágenes")
print(f"Prueba        : {len(test_df)} imágenes")

Mapeo de etiquetas: {'1-Cardboard': 0, '2-Food Organics': 1, '3-Glass': 2, '4-Metal': 3, '5-Miscellaneous Trash': 4, '6-Paper': 5, '7-Plastic': 6, '8-Textile Trash': 7, '9-Vegetation': 8}

Entrenamiento : 3325 imágenes
Validación    : 713 imágenes
Prueba        : 713 imágenes


## 1.2 Tratamiento del Desbalance de clases

In [8]:
from sklearn.utils.class_weight import compute_class_weight

# Calcular pesos de clase para compensar el desbalance
train_labels = train_df['label'].map(label_to_index).values
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=train_labels
)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}

print("Pesos por clase:")
for label, idx in sorted(label_to_index.items(), key=lambda x: x[1]):
    print(f"  {label:25s} (clase {idx}): {class_weight_dict[idx]:.4f}")

Pesos por clase:
  1-Cardboard               (clase 0): 1.1473
  2-Food Organics           (clase 1): 1.2828
  3-Glass                   (clase 2): 1.2566
  4-Metal                   (clase 3): 0.6681
  5-Miscellaneous Trash     (clase 4): 1.0678
  6-Paper                   (clase 5): 1.0556
  7-Plastic                 (clase 6): 0.5728
  8-Textile Trash           (clase 7): 1.6642
  9-Vegetation              (clase 8): 1.2113


In [9]:
def load_and_preprocess(image_path, label):
    """Lee una imagen desde disco, la redimensiona y normaliza a [0, 1]."""
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

def build_dataset(dataframe, shuffle=True):
    """Construye un tf.data.Dataset a partir de un DataFrame con columnas image_path y label."""
    paths = dataframe['image_path'].values
    labels = dataframe['label'].map(label_to_index).values
    labels = tf.keras.utils.to_categorical(labels, NUM_CLASSES)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(buffer_size=1024, seed=SEED)

    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

# Construcción de los pipelines
train_ds = build_dataset(train_df, shuffle=True)
val_ds   = build_dataset(val_df, shuffle=False)
test_ds  = build_dataset(test_df, shuffle=False)

# Verificación rápida
for images, labels in train_ds.take(1):
    print(f"Forma del lote de imágenes : {images.shape}")
    print(f"Forma del lote de etiquetas: {labels.shape}")

I0000 00:00:1779925990.040706      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779925990.047040      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Forma del lote de imágenes : (32, 299, 299, 3)
Forma del lote de etiquetas: (32, 9)


# 2. Transfer Learning — Búsqueda de Hiperparámetros (InceptionV3)

Se explora una rejilla de hiperparámetros del clasificador sobre una base **InceptionV3 completamente congelada**:

| Hiperparámetro | Valores |
|---|---|
| Learning rate | 1 × 10⁻³, 3 × 10⁻⁴, 1 × 10⁻⁴ |
| Capas densas del head | 1, 2 |
| Unidades por capa | 256, 512 |
| Dropout rate | 0.30, 0.50 |

**Total de combinaciones**: 3 × 2 × 2 × 2 = 24

In [10]:
# ── Espacio de búsqueda ──────────────────────────────────────────────────────
TL_LR_GRID       = [1e-3, 3e-4, 1e-4]   # 3 variaciones del learning rate
TL_DENSE_LAYERS  = [1, 2]                # 2 variaciones de capas densas
TL_UNITS_GRID    = [256, 512]            # 2 variaciones de unidades por capa
TL_DROPOUT_GRID  = [0.3, 0.5]           # 2 variaciones del dropout rate
TL_SEARCH_EPOCHS = 8                     # épocas por ensayo (base congelada → rápido)
TL_PATIENCE      = 3

def build_tl_model(learning_rate, num_dense_layers, units, dropout_rate):
    """InceptionV3 con base congelada y clasificador personalizado."""
    base = tf.keras.applications.InceptionV3(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    # Escalar [0, 1] → [–1, 1] según la normalización esperada por InceptionV3
    x = tf.keras.layers.Rescaling(scale=2.0, offset=-1.0)(inputs)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    for _ in range(num_dense_layers):
        x = tf.keras.layers.Dense(units, activation='relu')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(dropout_rate)(x)

    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

tl_grid = list(itertools.product(TL_LR_GRID, TL_DENSE_LAYERS, TL_UNITS_GRID, TL_DROPOUT_GRID))
print(f"Total combinaciones TL: {len(tl_grid)}")
# Verificar que el modelo se construye correctamente
_test = build_tl_model(1e-3, 1, 256, 0.3)
print(f"Parámetros entrenables (head): {_test.trainable_variables.__len__()}")
del _test

Total combinaciones TL: 24
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Parámetros entrenables (head): 6


In [11]:
## 2.1 Bucle de búsqueda
tl_results = []
tl_early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=TL_PATIENCE, restore_best_weights=True
)

for i, (lr, ndl, units, dr) in enumerate(tl_grid, 1):
    print(
        f"[{i:02d}/{len(tl_grid)}] lr={lr:.0e}  layers={ndl}  units={units}  dropout={dr}",
        end="  "
    )
    model = build_tl_model(lr, ndl, units, dr)
    hist = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=TL_SEARCH_EPOCHS,
        callbacks=[tl_early_stop],
        class_weight=class_weight_dict,
        verbose=0
    )
    best_val_acc  = max(hist.history['val_accuracy'])
    best_val_loss = min(hist.history['val_loss'])
    tl_results.append({
        'learning_rate'   : lr,
        'num_dense_layers': ndl,
        'units_per_layer' : units,
        'dropout_rate'    : dr,
        'val_accuracy'    : best_val_acc,
        'val_loss'        : best_val_loss,
        'epochs_ran'      : len(hist.history['val_accuracy'])
    })
    print(f"→ val_acc={best_val_acc:.4f}")
    del model  # liberar memoria GPU/CPU

tl_results_df = (
    pd.DataFrame(tl_results)
    .sort_values('val_accuracy', ascending=False)
    .reset_index(drop=True)
)

[01/24] lr=1e-03  layers=1  units=256  dropout=0.3  

I0000 00:00:1779926005.370056     133 service.cc:152] XLA service 0x7d1508003070 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779926005.370109     133 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779926005.370115     133 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779926007.812822     133 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1779926017.977807     133 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


→ val_acc=0.8022
[02/24] lr=1e-03  layers=1  units=256  dropout=0.5  → val_acc=0.8036
[03/24] lr=1e-03  layers=1  units=512  dropout=0.3  → val_acc=0.8022
[04/24] lr=1e-03  layers=1  units=512  dropout=0.5  → val_acc=0.7980
[05/24] lr=1e-03  layers=2  units=256  dropout=0.3  → val_acc=0.7770
[06/24] lr=1e-03  layers=2  units=256  dropout=0.5  → val_acc=0.7518
[07/24] lr=1e-03  layers=2  units=512  dropout=0.3  → val_acc=0.7854
[08/24] lr=1e-03  layers=2  units=512  dropout=0.5  → val_acc=0.7826
[09/24] lr=3e-04  layers=1  units=256  dropout=0.3  → val_acc=0.7882
[10/24] lr=3e-04  layers=1  units=256  dropout=0.5  → val_acc=0.7812
[11/24] lr=3e-04  layers=1  units=512  dropout=0.3  → val_acc=0.7952
[12/24] lr=3e-04  layers=1  units=512  dropout=0.5  → val_acc=0.7518
[13/24] lr=3e-04  layers=2  units=256  dropout=0.3  → val_acc=0.7784
[14/24] lr=3e-04  layers=2  units=256  dropout=0.5  → val_acc=0.7391
[15/24] lr=3e-04  layers=2  units=512  dropout=0.3  → val_acc=0.7784
[16/24] lr=3e-04 

In [12]:
## 2.2 Resultados y selección del mejor modelo TL
print("── Resultados búsqueda Transfer Learning (ordenados por val_accuracy) ──")
display(tl_results_df)

best_tl_cfg = tl_results_df.iloc[0]
BEST_TL_LR     = best_tl_cfg['learning_rate']
BEST_TL_LAYERS = int(best_tl_cfg['num_dense_layers'])
BEST_TL_UNITS  = int(best_tl_cfg['units_per_layer'])
BEST_TL_DR     = best_tl_cfg['dropout_rate']

print(f"\n★ Mejor configuración TL:")
print(f"  learning_rate    : {BEST_TL_LR:.0e}")
print(f"  num_dense_layers : {BEST_TL_LAYERS}")
print(f"  units_per_layer  : {BEST_TL_UNITS}")
print(f"  dropout_rate     : {BEST_TL_DR}")
print(f"  val_accuracy     : {best_tl_cfg['val_accuracy']:.4f}")

── Resultados búsqueda Transfer Learning (ordenados por val_accuracy) ──


,learning_rate,num_dense_layers,units_per_layer,dropout_rate,val_accuracy,val_loss,epochs_ran
0,0.0010,1,256,0.5,0.803647,0.615054,6
1,0.0010,1,256,0.3,0.802244,0.608438,5
2,0.0010,1,512,0.3,0.802244,0.627229,3
3,0.0003,2,512,0.5,0.802244,0.652259,3
4,0.0010,1,512,0.5,0.798036,0.653096,3
5,0.0003,1,512,0.3,0.795231,0.621457,3
6,0.0003,1,256,0.3,0.788219,0.616473,3
7,0.0010,2,512,0.3,0.785414,0.671668,3
8,0.0010,2,512,0.5,0.782609,0.672175,3
9,0.0001,1,512,0.3,0.782609,0.667878,3



★ Mejor configuración TL:
  learning_rate    : 1e-03
  num_dense_layers : 1
  units_per_layer  : 256
  dropout_rate     : 0.5
  val_accuracy     : 0.8036


In [13]:
## 2.3 Entrenamiento del modelo TL final (mejor configuración, más épocas)
TL_FINAL_EPOCHS = 20

best_tl_model = build_tl_model(BEST_TL_LR, BEST_TL_LAYERS, BEST_TL_UNITS, BEST_TL_DR)

tl_final_cbs = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1
    ),
]

print("Entrenando modelo TL final con la mejor configuración…")
tl_final_history = best_tl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=TL_FINAL_EPOCHS,
    callbacks=tl_final_cbs,
    class_weight=class_weight_dict,
    verbose=1
)

best_tl_model.save('best_tl_model.keras')
print("\nModelo TL guardado en 'best_tl_model.keras'")

Entrenando modelo TL final con la mejor configuración…
Epoch 1/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 43s 289ms/step - accuracy: 0.6247 - loss: 1.1068 - val_accuracy: 0.7027 - val_loss: 0.8558 - learning_rate: 0.0010
Epoch 2/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 15s 139ms/step - accuracy: 0.7835 - loss: 0.5626 - val_accuracy: 0.7658 - val_loss: 0.6822 - learning_rate: 0.0010
Epoch 3/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 15s 139ms/step - accuracy: 0.8307 - loss: 0.4555 - val_accuracy: 0.7391 - val_loss: 0.8011 - learning_rate: 0.0010
Epoch 4/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 15s 138ms/step - accuracy: 0.8469 - loss: 0.3925 - val_accuracy: 0.7812 - val_loss: 0.6435 - learning_rate: 0.0010
Epoch 5/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 15s 138ms/step - accuracy: 0.8752 - loss: 0.3280 - val_accuracy: 0.7952 - val_loss: 0.6412 - learning_rate: 0.0010
Epoch 6/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 15s 137ms/step - accuracy: 0.8884 - loss: 0.2987 - val_accuracy: 0.8036 - val_loss: 0.5617 - learning_rate: 0.0010
Epoch 7/20
104/

# 3. Fine-Tuning — Búsqueda de Hiperparámetros

Partiendo del **modelo TL guardado** (arquitectura y pesos fijados), se descongela progresivamente el extremo superior del backbone InceptionV3. Se evita descongelar todo el backbone para no sobreajustar dado el tamaño del conjunto de datos.

| Hiperparámetro | Valores |
|---|---|
| Learning rate | 1 × 10⁻⁴, 5 × 10⁻⁵, 1 × 10⁻⁵ |
| Capas del backbone a descongelar | 20, 50 |

**Total de combinaciones**: 3 × 2 = 6

In [14]:
# ── Espacio de búsqueda – Fine-Tuning ────────────────────────────────────────
FT_LR_GRID       = [1e-4, 5e-5, 1e-5]  # 3 variaciones (learning rates bajos)
FT_UNFREEZE_GRID = [20, 50]             # 2 variaciones de capas descongeladas
FT_EPOCHS        = 10
FT_PATIENCE      = 4
TL_MODEL_PATH    = 'best_tl_model.keras'

def apply_finetune(model_path, learning_rate, layers_to_unfreeze):
    """
    Carga el modelo TL guardado, descongela las últimas `layers_to_unfreeze`
    capas del backbone InceptionV3 y recompila con un learning rate reducido.
    El resto del backbone permanece congelado.
    """
    model = tf.keras.models.load_model(model_path)

    # Localizar el backbone (único submodelo en la arquitectura)
    base = next(layer for layer in model.layers if isinstance(layer, tf.keras.Model))

    # Descongelar solo las últimas N capas; el resto sigue congelado
    base.trainable = True
    for layer in base.layers[:-layers_to_unfreeze]:
        layer.trainable = False
    for layer in base.layers[-layers_to_unfreeze:]:
        layer.trainable = True

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

ft_grid = list(itertools.product(FT_LR_GRID, FT_UNFREEZE_GRID))
print(f"Total combinaciones FT: {len(ft_grid)}")

Total combinaciones FT: 6


In [15]:
## 3.1 Bucle de búsqueda – Fine-Tuning
ft_results = []
ft_early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=FT_PATIENCE, restore_best_weights=True
)

for i, (lr, unfreeze) in enumerate(ft_grid, 1):
    print(
        f"[{i}/{len(ft_grid)}] lr={lr:.0e}  capas_descongeladas={unfreeze}",
        end="  "
    )
    model = apply_finetune(TL_MODEL_PATH, lr, unfreeze)
    hist = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=FT_EPOCHS,
        callbacks=[ft_early_stop],
        class_weight=class_weight_dict,
        verbose=0
    )
    best_val_acc  = max(hist.history['val_accuracy'])
    best_val_loss = min(hist.history['val_loss'])
    ft_results.append({
        'learning_rate'  : lr,
        'layers_unfrozen': unfreeze,
        'val_accuracy'   : best_val_acc,
        'val_loss'       : best_val_loss,
        'epochs_ran'     : len(hist.history['val_accuracy'])
    })
    print(f"→ val_acc={best_val_acc:.4f}")
    del model  # liberar memoria

ft_results_df = (
    pd.DataFrame(ft_results)
    .sort_values('val_accuracy', ascending=False)
    .reset_index(drop=True)
)

[1/6] lr=1e-04  capas_descongeladas=20  → val_acc=0.8738
[2/6] lr=1e-04  capas_descongeladas=50  → val_acc=0.8780
[3/6] lr=5e-05  capas_descongeladas=20  → val_acc=0.8415
[4/6] lr=5e-05  capas_descongeladas=50  → val_acc=0.8597
[5/6] lr=1e-05  capas_descongeladas=20  → val_acc=0.8359
[6/6] lr=1e-05  capas_descongeladas=50  → val_acc=0.8149


In [16]:
## 3.2 Resultados y entrenamiento del modelo FT final
print("── Resultados búsqueda Fine-Tuning (ordenados por val_accuracy) ──")
display(ft_results_df)

best_ft_cfg      = ft_results_df.iloc[0]
BEST_FT_LR       = best_ft_cfg['learning_rate']
BEST_FT_UNFREEZE = int(best_ft_cfg['layers_unfrozen'])

print(f"\n★ Mejor configuración Fine-Tuning:")
print(f"  learning_rate    : {BEST_FT_LR:.0e}")
print(f"  layers_unfrozen  : {BEST_FT_UNFREEZE}")
print(f"  val_accuracy     : {best_ft_cfg['val_accuracy']:.4f}")

# Modelo FT final con más épocas a partir del checkpoint TL
FT_FINAL_EPOCHS = 20
best_ft_model   = apply_finetune(TL_MODEL_PATH, BEST_FT_LR, BEST_FT_UNFREEZE)

ft_final_cbs = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-8, verbose=1
    ),
]

print("\nEntrenando modelo FT final…")
ft_final_history = best_ft_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FT_FINAL_EPOCHS,
    callbacks=ft_final_cbs,
    class_weight=class_weight_dict,
    verbose=1
)

best_ft_model.save('best_ft_model.keras')
print("\nModelo FT guardado en 'best_ft_model.keras'")

── Resultados búsqueda Fine-Tuning (ordenados por val_accuracy) ──


,learning_rate,layers_unfrozen,val_accuracy,val_loss,epochs_ran
0,0.00010,50,0.877980,0.381412,7
1,0.00010,20,0.873773,0.422666,10
2,0.00005,50,0.859748,0.454308,4
3,0.00005,20,0.841515,0.509873,4
4,0.00001,20,0.835905,0.524845,4
5,0.00001,50,0.814867,0.575018,4



★ Mejor configuración Fine-Tuning:
  learning_rate    : 1e-04
  layers_unfrozen  : 50
  val_accuracy     : 0.8780

Entrenando modelo FT final…
Epoch 1/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 56s 342ms/step - accuracy: 0.8722 - loss: 0.3440 - val_accuracy: 0.8163 - val_loss: 0.6931 - learning_rate: 1.0000e-04
Epoch 2/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 18s 161ms/step - accuracy: 0.9657 - loss: 0.1026 - val_accuracy: 0.8499 - val_loss: 0.4744 - learning_rate: 1.0000e-04
Epoch 3/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 18s 162ms/step - accuracy: 0.9877 - loss: 0.0475 - val_accuracy: 0.8597 - val_loss: 0.4381 - learning_rate: 1.0000e-04
Epoch 4/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 17s 162ms/step - accuracy: 0.9925 - loss: 0.0311 - val_accuracy: 0.8682 - val_loss: 0.4479 - learning_rate: 1.0000e-04
Epoch 5/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 17s 158ms/step - accuracy: 0.9955 - loss: 0.0175 - val_accuracy: 0.8640 - val_loss: 0.4175 - learning_rate: 1.0000e-04
Epoch 6/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 17s 160ms/step - accu